[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Qwen35_0p8B_V4_DeltaOnly_Isolation_Colab.ipynb)

# Qwen3.5-0.8B V4 — DeltaNet-only replacement

## Purpose
This isolates your window+anchor approach in **one mixer family only**, holding the other mixer family exactly as pretrained. Both V4 notebooks share the same deterministic seed, official WikiText TRAIN / VALIDATION / TEST subsets, next-token teacher agreement, JS, sampled next-token NLL ratio, and no-cache FP16 latency at 64/128/192 tokens.

**Delta-only:** The 18 native Gated DeltaNet blocks are tested one at a time, then replaced progressively (1, 3, 6, 12, 18). The six original full-attention blocks are untouched. The best DEV subset is selected. Optional 120-step calibration distillation trains only new window/anchor gain, bias, temperature and decay, with rollback to the best DEV checkpoint.**

All comparisons are made against the original unmodified teacher. The holdout remains sealed until after layer ordering and the optional Delta distillation. The final custom model is tested by saving/reloading its exact selected topology and comparing three independent DEV logits.

**Outputs:** An automatically downloaded statistics ZIP, plus an optional full FP16 model ZIP; a private Hugging Face upload is attempted only if holdout quality, chat quality, latency and strict reload checks all pass. Failure also produces a diagnostic ZIP.

**Limitations:** Text-only, full-prefix no-cache inference. These notebooks do NOT support incremental KV-cache generation and do not establish real-time streaming speed.


In [ ]:
#@title 1. Install compatible Qwen3.5 runtime + isolated V4 experiment
%pip -q install -U "git+https://github.com/huggingface/transformers.git@main" "datasets>=3" "accelerate>=1.0" "huggingface_hub>=0.34" "safetensors>=0.5"
%pip -q install --no-deps "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@8aaa783d894974637c9225f55c7bd0a8c79babb7"
from tinycenn_lm.qwen35_isolated_runner import run
MODE="delta"
print("Ready: Qwen3.5 0.8B", MODE, "only. The other attention family is not modified.")


In [ ]:
#@title 2. Run independent delta-only experiment — downloads final statistics ZIP
DELTA_WINDOW=128 #@param {type:"integer"}
FULL_WINDOW=512 #@param {type:"integer"}
ANCHORS=16 #@param {type:"integer"}
CHUNK_SIZE=64 #@param {type:"integer"}
MAX_CONTEXT=192 #@param {type:"integer"}
TRAIN_N=96 #@param {type:"integer"}
DEV_N=48 #@param {type:"integer"}
HOLDOUT_N=96 #@param {type:"integer"}
FINETUNE_SELECTED=True #@param {type:"boolean"}
FINETUNE_STEPS=120 #@param {type:"integer"}
SAVE_LOCAL_MODEL=True #@param {type:"boolean"}
DOWNLOAD_STATS=True #@param {type:"boolean"}
DOWNLOAD_MODEL_ZIP=False #@param {type:"boolean"}
UPLOAD_HF_IF_QUALIFIED=True #@param {type:"boolean"}
HF_PRIVATE=True #@param {type:"boolean"}

RUN_RESULTS=run(mode=MODE,seed=20261019,
    delta_window=DELTA_WINDOW,full_window=FULL_WINDOW,
    anchors=ANCHORS,chunk_size=CHUNK_SIZE,max_context=MAX_CONTEXT,
    train_n=TRAIN_N,dev_n=DEV_N,holdout_n=HOLDOUT_N,
    train_delta=FINETUNE_SELECTED,train_steps=FINETUNE_STEPS,
    save_model=SAVE_LOCAL_MODEL,
    download_stats=DOWNLOAD_STATS,download_model=DOWNLOAD_MODEL_ZIP,
    upload_if_good=UPLOAD_HF_IF_QUALIFIED,hf_private=HF_PRIVATE)
print("Completed diagnostics:",RUN_RESULTS.get("qualified"),
      "HF publication only if qualification and model reload pass")
